In [210]:
import pandas as pd

data = pd.read_csv("../../data/raw/kathmandu_full_raw_2023_2024.csv")

Datetime format conversion and sorting (although already sorted)

In [211]:

data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").reset_index(drop=True)

Create target column. For instance i, target is PM2.5 of instance (i+1) i.e. PM2.5 after 1 hour.

In [212]:
data["target"] = data["pm2_5"].shift(-1)
data.head(3)

,time,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,surface_pressure,target
0,2023-01-01 00:00:00,83.4,119.2,1689,38.0,12.9,67,7.9,88,1.1,162,872.6,82.2
1,2023-01-01 01:00:00,82.2,117.6,1594,31.5,13.2,70,8.0,85,2.9,150,872.3,80.8
2,2023-01-01 02:00:00,80.8,115.7,1504,24.6,13.6,73,8.0,81,4.1,135,871.9,78.2


PM2.5 lag features. i.e. PM2.5 at (t-1), (t-2), ... (t-12)

In [213]:
for i in range(1, 13):
    data[f"pm2_5_lag_{i}"] = data["pm2_5"].shift(i)

data[20:25]

,time,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,temperature_2m,relative_humidity_2m,wind_speed_10m,...,pm2_5_lag_3,pm2_5_lag_4,pm2_5_lag_5,pm2_5_lag_6,pm2_5_lag_7,pm2_5_lag_8,pm2_5_lag_9,pm2_5_lag_10,pm2_5_lag_11,pm2_5_lag_12
20,2023-01-01 20:00:00,88.2,127.1,1713,52.2,11.1,62,9.2,81,3.1,...,66.8,63.8,62.4,62.2,64.0,69.2,69.8,70.3,71.5,88.5
21,2023-01-01 21:00:00,95.2,136.9,1845,54.2,11.4,56,8.3,85,3.6,...,75.0,66.8,63.8,62.4,62.2,64.0,69.2,69.8,70.3,71.5
22,2023-01-01 22:00:00,96.6,138.8,1921,53.5,11.8,52,7.4,89,2.0,...,81.0,75.0,66.8,63.8,62.4,62.2,64.0,69.2,69.8,70.3
23,2023-01-01 23:00:00,93.1,134.1,1918,48.8,12.1,53,6.7,92,2.0,...,88.2,81.0,75.0,66.8,63.8,62.4,62.2,64.0,69.2,69.8
24,2023-01-02 00:00:00,90.3,129.9,1795,41.6,12.6,59,6.0,93,2.5,...,95.2,88.2,81.0,75.0,66.8,63.8,62.4,62.2,64.0,69.2


Pollutant lag features, for pollutants, (t-1), (t-2), (t-3).

In [214]:
pollutants = ["pm10", "carbon_monoxide", "nitrogen_dioxide", "sulphur_dioxide", "ozone"]

for col in pollutants:
    for i in range(1, 4):
        data[f"{col}_lag_{i}"] = data[col].shift(i)

Meterological lag features, (t-1).

In [215]:
meteo = ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "wind_direction_10m", "surface_pressure"]

for col in meteo:
    data[f"{col}_lag_1"] = data[col].shift(1)

Handling null values

In [216]:
data = data.dropna().reset_index(drop=True)

In [222]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 17531 entries, 0 to 17530
Data columns (total 45 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   time                        17531 non-null  datetime64[us]
 1   pm2_5                       17531 non-null  float64       
 2   pm10                        17531 non-null  float64       
 3   carbon_monoxide             17531 non-null  int64         
 4   nitrogen_dioxide            17531 non-null  float64       
 5   sulphur_dioxide             17531 non-null  float64       
 6   ozone                       17531 non-null  int64         
 7   temperature_2m              17531 non-null  float64       
 8   relative_humidity_2m        17531 non-null  int64         
 9   wind_speed_10m              17531 non-null  float64       
 10  wind_direction_10m          17531 non-null  int64         
 11  surface_pressure            17531 non-null  float64       
 12  t

Meterological bin features. (based on meterological analysis in EDA)

In [207]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class MeterologicalBinFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X["temp_bin"] = pd.cut(
            data["temperature_2m"],
            bins=[-np.inf, 10, 20, np.inf],
            labels=["low", "medium", "high"]
        )

        X["humidity_bin"] = pd.cut(
            data["relative_humidity_2m"],
            bins=[-np.inf, 40, np.inf],
            labels=["low", "normal"]
        )

        X["wind_speed_bin"] = pd.cut(
            data["wind_speed_10m"],
            bins=[-np.inf, 3, 6, 12, 15, np.inf],
            labels=["calm", "light", "moderate", "strong", "very_strong"]
        )

        X["wind_direction_bin"] = np.where(
            (data["wind_direction_10m"] > 120) & (data["wind_direction_10m"] <= 240),
            "mid",
            "outer"
        )

        X["surface_pressure_bin"] = pd.cut(
            data["surface_pressure"],
            bins=[-np.inf, 865, 870, 875, np.inf],
            labels=["very_low", "low", "normal", "high"]
        )

        return X

Temporal bin features. (based on temporal analysis in EDA)

In [208]:
class TemporalBinFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        X["hour"] = X["time"].dt.hour
        X["day_of_week"] = X["time"].dt.dayofweek
        X["month"] = X["time"].dt.month

        X["season"] = np.where(
            data["month"].isin([11, 12, 1, 2]),
            "Nov_Feb",
            np.where(
                data["month"].isin([3, 4, 5, 6]),
                "Mar_Jun",
                "Jul_Oct"
            )
        )

        X["time_of_day"] = np.where(
            data["hour"].isin([19, 20, 21, 22, 23, 0]),
            "7pm_to_12am",
            np.where(
                data["hour"].isin([1, 2, 3, 4, 5, 6, 7, 8]),
                "1am_to_8am",
                np.where(
                    data["hour"].isin([9, 10, 11, 12, 13, 14, 15, 16]),
                    "9am_to_4pm",
                    "5pm_to_6pm"
                )
            )
        )
        
        return X


Cycle encoding for hour, day and month (raw numeric values can negatively affect some models)

In [ ]:
class CyclicalFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # cyclical features
        X["hour_sin"] = np.sin(2*np.pi*X["hour"]/24)
        X["hour_cos"] = np.cos(2*np.pi*X["hour"]/24)

        X["dow_sin"] = np.sin(2*np.pi*X["day_of_week"]/7)
        X["dow_cos"] = np.cos(2*np.pi*X["day_of_week"]/7)

        X["month_sin"] = np.sin(2*np.pi*X["month"]/12)
        X["month_cos"] = np.cos(2*np.pi*X["month"]/12)

        X = X.drop(columns=["hour", "day_of_week", "month"])

        return X

One-hot encoding for categorical columns.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

categorical_cols = [
    "temp_bin",
    "humidity_bin",
    "wind_speed_bin",
    "wind_direction_bin",
    "surface_pressure_bin",
    "season",
    "time_of_day",
]

pipeline = Pipeline([
    ("meterological", MeterologicalBinFeatures()),
    ("temporal", TemporalBinFeatures()),
    ("cyclical", CyclicalFeatures()),
    ("preprocess", ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
        ],
        remainder="passthrough"
    )),
    ("scaler", StandardScaler())
])
